# What do team scoring profiles capture?

An analyst reviewing completed Premier League seasons may want a compact comparison of team attacking and defending rates by venue. A profile is one club-season with four mean goals-per-match values. These are retrospective outcomes from the public-domain openfootball archive retrieved November 20, 2026. See the dataset card for checksums. Download the development seasons with `uv run python scripts/download_football.py` before running from a clean kernel.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from dsm_project.config import load_config
from dsm_project.data.openfootball import load_matches
from dsm_project.experiments.team_profiles import profile_experiment, team_season_profiles

root = Path.cwd() if (Path.cwd() / 'configs').is_dir() else Path.cwd().parent
matches = load_matches(load_config(root / 'configs/football.toml'))
rates = team_season_profiles(matches)
display(rates.describe().round(2))
ax = rates.plot.scatter(x='away_attack', y='home_attack', alpha=0.65, color='#287a5a')
ax.set(xlabel='Away goals scored per match', ylabel='Home goals scored per match')
plt.show()

Teams scoring more away also tend to score more at home. Standardizing each rate gives it comparable scale. PCA finds orthogonal directions of maximum sample variance and projects each standardized row onto the two leading directions. K-means minimizes the sum of squared Euclidean distances to the nearest centroid in all four standardized dimensions. Start from ten initializations per seed. These are scikit-learn references linked through reusable package functions, with verified archive structure and a hand-checkable aggregation test. Student implementations are pending.

Use silhouette to compare two to five clusters and adjusted Rand index to check seed sensitivity against the reference run. PCA reconstruction error measures distortion from dropping two directions. This descriptive fit uses all six development seasons, and repeated clubs can make apparent structure more stable than it would be in a future season.

In [ ]:
profiles, evidence = profile_experiment(matches)
display(pd.DataFrame(evidence['candidates']))
display(pd.DataFrame(evidence['pca_loadings'], index=['PC 1', 'PC 2']).T)
print('Explained variance', evidence['explained_variance'])
print('Reconstruction RMSE in standardized units', evidence['reconstruction_rmse'])
for cluster, group in profiles.groupby('cluster'):
    plt.scatter(group.pc1, group.pc2, label=f'Cluster {cluster}', alpha=0.65)
plt.xlabel('First principal component (standardized score)')
plt.ylabel('Second principal component (standardized score)')
plt.legend()
plt.show()

Two components explain about 82% of variation and leave about 0.42 standardized units of reconstruction RMSE. In this run, higher PC 1 corresponds to more scoring and fewer goals conceded, so the separation largely reflects strength. Two clusters had the highest silhouette, about 0.35, and identical assignments in the five seed comparisons. The modest silhouette and repeated clubs limit interpretation. A held-back season or resampling whole clubs would test generalization. Keep this as descriptive evidence alongside the live form comparison.